In [1]:
import sqlite3
import pandas as pd

In [2]:
# 1. Connect to SQLite database (creates the file if it doesn't exist)
conn = sqlite3.connect("sports_team.db")
cursor = conn.cursor()

# Enable foreign key support (good practice, though not explicitly required here)
cursor.execute("PRAGMA foreign_keys = ON;")

In [3]:
# 2. Create the table with the required SQL constraints
cursor.execute("""
CREATE TABLE IF NOT EXISTS players (
    player_id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT NOT NULL,
    jersey_number INTEGER NOT NULL UNIQUE,
    position TEXT DEFAULT 'Undeclared',
    email TEXT UNIQUE
);
""")
conn.commit()
print("Table 'players' created successfully with constraints.")

Table 'players' created successfully with constraints.


In [4]:
# 3. Helper function to insert players safely and test invalid entries
def insert_player(name, jersey_number, position=None, email=None):
    try:
        # Build query based on whether optional parameters are provided to allow DEFAULTs to work
        if position is None:
            cursor.execute(
                "INSERT INTO players (name, jersey_number, email) VALUES (?, ?, ?)",
                (name, jersey_number, email)
            )
        else:
            cursor.execute(
                "INSERT INTO players (name, jersey_number, position, email) VALUES (?, ?, ?, ?)",
                (name, jersey_number, position, email)
            )
        conn.commit()
        print(f"Successfully added player: {name}")
    except sqlite3.IntegrityError as e:
        print(f"❌ Database Constraint Error adding '{name}': {e}")


print("\n--- Inserting Valid Records ---")
insert_player("Alice Smith", 10, "Forward", "alice@team.com")
insert_player("Bob Jones", 23, "Goalkeeper", "bob@team.com")
# Testing DEFAULT value for position and NULL value for email
insert_player("Charlie Brown", 7, email=None) 

print("\n--- Testing Invalid Entries (Testing Constraints) ---")



--- Inserting Valid Records ---
Successfully added player: Alice Smith
Successfully added player: Bob Jones
Successfully added player: Charlie Brown

--- Testing Invalid Entries (Testing Constraints) ---


In [5]:
# 1. Test NOT NULL constraint (Missing Name)
insert_player(None, 44, "Defender", "none@team.com")

❌ Database Constraint Error adding 'None': NOT NULL constraint failed: players.name


In [6]:
# 2. Test UNIQUE constraint (Duplicate Jersey Number)
insert_player("Duplicate Jersey Guy", 10, "Midfielder", "dup1@team.com")

❌ Database Constraint Error adding 'Duplicate Jersey Guy': UNIQUE constraint failed: players.jersey_number


In [7]:
# 3. Test UNIQUE constraint (Duplicate Email)
insert_player("Duplicate Email Guy", 99, "Forward", "alice@team.com")

❌ Database Constraint Error adding 'Duplicate Email Guy': UNIQUE constraint failed: players.email


In [8]:
# 4. Explore the database using Pandas
print("\n--- Exploring Database Data with Pandas ---")
df = pd.read_sql_query("SELECT * FROM players", conn)
print(df)


--- Exploring Database Data with Pandas ---
   player_id           name  jersey_number    position           email
0          1    Alice Smith             10     Forward  alice@team.com
1          2      Bob Jones             23  Goalkeeper    bob@team.com
2          3  Charlie Brown              7  Undeclared             NaN


In [9]:
# Close the database connection
conn.close()